# 04 - Statistical Analysis
In this notebook, we perform fundamental statistical tests to validate insights derived from the EDA. We will focus on widely-used methods like Correlation Analysis, Difference Testing (T-Test), ANOVA (Multivariate Analysis for groups), and Chi-Square Testing.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, ttest_ind, f_oneway, pearsonr
import warnings

warnings.filterwarnings('ignore')

# Helper function for Cramer's V (Effect Size for Chi-Square)
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))


## 1. Load Data


In [ ]:
cleaned_df = pd.read_csv('../data/processed/cleaned_dataset.csv')

# Create binary column for cancellation tests
cleaned_df['Is_Cancelled'] = cleaned_df['Booking_Status'].apply(lambda x: 0 if x == 'Success' else 1)
print("Data loaded successfully. Total rows:", len(cleaned_df))


## 2. Correlation Analysis
Using Pearson Correlation to strictly quantify the linear relationship between continuous variables (like Wait Time) and Cancellation.


In [ ]:
numeric_cols = ['Ride_Distance', 'Booking_Value', 'V_TAT', 'Is_Cancelled']
# Drop NaNs to perform correlation
corr_data = cleaned_df[numeric_cols].dropna()

print("Pearson Correlation with Cancellation (Is_Cancelled):")
for col in ['Ride_Distance', 'Booking_Value', 'V_TAT']:
    corr, p_val = pearsonr(corr_data[col], corr_data['Is_Cancelled'])
    print(f"- {col}: Correlation = {corr:.4f}, P-value = {p_val:.4e}")

print("\nInterpretation: A positive correlation with V_TAT means that as wait times increase, the likelihood of cancellation also increases.")


## 3. Difference Testing: Independent T-Test
We will use an Independent Two-Sample T-Test to see if there is a statistically significant difference in the *average* Wait Time (V_TAT) between Successful rides and Cancelled rides.


In [ ]:
# Separate V_TAT for success vs cancelled
success_vtat = cleaned_df[cleaned_df['Is_Cancelled'] == 0]['V_TAT'].dropna()
cancelled_vtat = cleaned_df[cleaned_df['Is_Cancelled'] == 1]['V_TAT'].dropna()

# Perform Independent T-Test
t_stat, p_value = ttest_ind(success_vtat, cancelled_vtat, equal_var=False) # Welch's T-test

print(f"Average Wait Time (Success): {success_vtat.mean():.2f} seconds")
print(f"Average Wait Time (Cancelled): {cancelled_vtat.mean():.2f} seconds")
print(f"\nT-Statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("\nResult: Reject the Null Hypothesis. There is a statistically significant difference in wait times between successful and cancelled rides.")
else:
    print("\nResult: Fail to reject Null Hypothesis.")


## 4. Multivariate Group Analysis: One-Way ANOVA
We will use a One-Way ANOVA to test if average Ride Distances differ significantly across multiple groups (the different Vehicle Types).


In [ ]:
# Group distances by vehicle type
vehicle_types = cleaned_df['Vehicle_Type'].dropna().unique()
distance_groups = [cleaned_df[cleaned_df['Vehicle_Type'] == vt]['Ride_Distance'].dropna() for vt in vehicle_types]

# Perform ANOVA
f_stat, p_val_anova = f_oneway(*distance_groups)

print(f"ANOVA F-Statistic: {f_stat:.4f}")
print(f"P-value: {p_val_anova:.4e}")

if p_val_anova < 0.05:
    print("\nResult: Reject the Null Hypothesis. Average ride distances vary significantly across different Vehicle Types.")
else:
    print("\nResult: Fail to reject Null Hypothesis. Ride distances do not vary significantly across Vehicle Types.")


## 5. Chi-Square Test & Effect Size
Testing if the categorical variable `Vehicle_Type` significantly impacts the categorical variable `Is_Cancelled`.


In [ ]:
contingency_vehicle = pd.crosstab(cleaned_df['Vehicle_Type'], cleaned_df['Is_Cancelled'])
chi2_v, p_val_v, dof_v, expected_v = chi2_contingency(contingency_vehicle)
effect_size_v = cramers_v(contingency_vehicle)

print(f"Chi-Square Statistic: {chi2_v:.4f}")
print(f"P-value: {p_val_v:.4e}")
print(f"Cramer's V (Effect Size): {effect_size_v:.4f}")

if p_val_v < 0.05:
    print("\nResult: Significant relationship found between Vehicle Type and Cancellation.")
    if effect_size_v < 0.1:
        print("Effect is weak.")
    elif effect_size_v < 0.3:
        print("Effect is moderate.")
    else:
        print("Effect is strong.")


## Conclusion & Summary of Findings

The fundamental statistical tests cleanly validate our visual findings:
- **Correlation & Difference (T-Test)**: We found a highly significant difference in wait times (`V_TAT`) between successful and cancelled rides. The positive Pearson correlation confirms that as wait times go up, cancellations follow.
- **Multivariate Analysis (ANOVA)**: The ANOVA test confirms that different vehicle types cater to fundamentally different trip lengths (Ride Distance).
- **Chi-Square**: Confirms that your choice of Vehicle Type has a statistically significant impact on whether the ride will be completed.
